[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/optics/coating/coating.ipynb)

# Optical coatings from constitutive maps

A material's constitutive map determines how light travels through it and what reflects at its faces. Here the same construction makes a matching film, a reflector with twisting optical axes, and an interface that converts polarization through an axion jump. The fields, material response, propagation, and reflection remain extensors throughout.

In [ ]:
# The repository root on the path, for numga and the examples; in Colab, fetch the repository first.
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    root = Path("/content/numga")
    if not root.exists():
        import subprocess
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/EelcoHoogendoorn/numga.git", str(root)], check=True)
else:
    root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "numga").is_dir() and (p / "examples").is_dir())
sys.path.insert(0, str(root))

In [ ]:
%matplotlib inline
from functools import reduce
from itertools import accumulate, islice

import numpy as np

from numga import NumpyContext, stack
from numga.algebras import STA
from numga.sparse import SparseExtensor
from examples.animation import save_figure, show_animation
from examples.optics.coating import render
from examples.optics.coating.core import Medium, Ports, Scattering

In [ ]:
# Time squares to plus one; the three spatial directions square to minus one.
ga = STA
mv = NumpyContext(ga, dtype=np.complex128).multivector
Scalar = ga.gatype.scalar()
Vector = ga.gatype.vector()
Bivector = ga.gatype.bivector()
Antibivector = ga.gatype.antibivector()
Polarization = ga.gatype(ga.subspace("x y"))
Boundary = ga.gatype(ga.subspace("x y txz tyz"))
Constitutive = ga.gatype((Antibivector, Bivector))
Propagation = ga.gatype((Boundary, Boundary))
Outgoing = ga.gatype((Boundary, Polarization))

# Tangential traces, expressed through the observer and face normal.
field_boundary = mv.t ^ (mv.t | (mv.z ^ Bivector))
excitation_boundary = mv.t | (mv.t ^ (mv.z ^ Antibivector).dual())

## 1. Let the material act on physical fields

A field bivector contains electric and magnetic planes in spacetime. Complex coefficients carry the phase of their harmonic oscillation; the positive imaginary phase increases with time. The constitutive map `chi` takes the field to its excitation. Weighting the three electric planes gives the principal permittivities, while the spatial planes receive the inverse permeability. These responses are in vacuum units.

In [ ]:
substrate_index = 1.5
permittivity = np.full(3, substrate_index ** 2)
reluctivity = 1.0

axes = stack([mv.x, mv.y, mv.z])


def dielectric(permittivity: np.ndarray, reluctivity: np.ndarray) -> Constitutive:
    planes = axes ^ mv.t
    # Leave the field open: each weighted plane contributes to the whole material map.
    electric_response = (permittivity * planes * (planes | Bivector)).sum(axis=-1)
    magnetic_response = (Bivector + (mv.t >> Bivector)) / 2
    return (electric_response + reluctivity * magnetic_response).dual()


substrate_chi = dielectric(permittivity, reluctivity)                # Antibivector <- Bivector



## 2. Recover the full field from its boundary amplitudes

At a face normal to `mv.z`, four combinations of field and excitation are continuous. They occupy the vector and trivector slots of `Boundary`. Two remaining Maxwell equations constrain the longitudinal fields. Solving these six conditions together reconstructs the full bivector from a boundary state. The temporal residual applied to that reconstruction gives the boundary's propagation generator.

In [ ]:
parallel = mv.t                                                    # Normal incidence.


def medium_from_chi(chi: Constitutive, parallel: Vector) -> Medium:
    boundary = field_boundary + excitation_boundary(chi)           # Boundary <- Bivector
    # Wedges and contractions isolate the two equations without normal derivatives.
    constraint = (-(mv.z | (mv.z ^ (parallel ^ Bivector)))
                  - mv.z * (mv.z | (parallel ^ chi).dual()))
    reconstruct = (boundary + constraint).solve(Boundary)           # Bivector <- Boundary
    # Apply Maxwell evolution, then take the same tangential traces.
    generator = (field_boundary(mv.z | (parallel ^ reconstruct))
                 + excitation_boundary(mv.z | (parallel ^ chi(reconstruct))))
    return Medium(boundary, reconstruct, generator)


substrate_medium = medium_from_chi(substrate_chi, parallel)

## 3. Propagate every mode and compose the layers

Each eigenmode acquires its own phase across a homogeneous layer. The modes need not be orthogonal: the sum of their dyads is a map, and solving it against each mode gives the reciprocal that reads that mode's amplitude and no other. Weight each mode by its phase and read it with its reciprocal to propagate the whole boundary state. The resulting map carries a film's right face back to its left; adjacent films compose in their physical order.

In [ ]:
def propagate(generator: Propagation, optical_distance: np.ndarray) -> Propagation:
    indices, modes = generator.eig()                              # [..., modes] Scalar, Boundary
    frame = (modes * modes.scalar_product(Boundary)).sum(axis=-1)  # [...] Boundary <- Boundary
    # Each reciprocal reads the amplitude of its own mode and of no other.
    reciprocal = frame[..., None].solve(modes)                     # [..., modes] Boundary
    # The phase increases when carrying a forward wave from right to left.
    phases = (indices * (1j * optical_distance[..., None])).exp()
    return (phases * modes * reciprocal.scalar_product(Boundary)).sum(axis=-1)


def compose(layers: Propagation) -> Propagation:
    return reduce(lambda propagation, layer: propagation(layer), layers)

## 4. Leave the incident polarization open

In each isotropic exterior, forward and backward waves have opposite normal propagation indices. Their electric amplitudes are read from those two parts of the boundary state. The substrate has only an outgoing wave. Carry it through the coating and solve for its incoming amplitude, leaving `Polarization` open: the result is the transmission map for every incident polarization, followed by the reflection map.

In [ ]:
# Undo the wedges carrying the electric part of the boundary state.
electric_trace = -mv.t | (mv.z | Boundary)        # Polarization <- Boundary


def ports_from_medium(medium: Medium, index: float, parallel: Vector) -> Ports:
    normal_index = (index ** 2 + (parallel - mv.t).squared()).square_root()
    # Complete the tangential electric field so that it is perpendicular to the ray.
    electric_field = Polarization + mv.z * (parallel | Polarization) / normal_index
    wavevector = parallel + mv.z * normal_index
    outgoing = medium.boundary(wavevector ^ electric_field)       # Boundary <- Polarization
    incoming = electric_trace((Boundary + medium.generator / normal_index) / 2)
    returning = electric_trace((Boundary - medium.generator / normal_index) / 2)
    return Ports(outgoing, incoming, returning, electric_field, normal_index)


def isotropic_ports(index: float, parallel: Vector) -> Ports:
    # A lossless, nonmagnetic isotropic exterior of the given refractive index.
    chi = dielectric(np.full(3, index ** 2), reluctivity)
    return ports_from_medium(medium_from_chi(chi, parallel), index, parallel)


def scatter(entrance: Outgoing, incident: Ports) -> Scattering:
    # The substrate's outgoing wave, carried back through any coating to the entrance face.
    transmission = incident.incoming(entrance).inverse()
    reflection = incident.returning(entrance)(transmission)
    return Scattering(reflection, transmission)

Power depends on both real quadratures of the electric field. Reversion acts on blades and leaves complex coefficients unconjugated, so a reverse product alone does not perform this phase average. Take the squared lengths of the real field and its quarter-cycle-shifted field, then multiply by the normal propagation index. The common averaging factor cancels in reflected and transmitted power ratios.

In [ ]:
def power(ports: Ports, polarization: Polarization) -> Scalar:
    field = ports.electric_field(polarization)
    # Spatial vectors have negative squared length in this signature.
    magnitude = -(field.real().scalar_norm_squared()
                  + (-1j * field).real().scalar_norm_squared())
    return ports.normal_index.real() * magnitude


def powers(result: Scattering, polarization: Polarization,
           incident: Ports, substrate: Ports) -> tuple[Scalar, Scalar]:
    normalization = power(incident, polarization)
    reflected = power(incident, result.reflection(polarization)) / normalization
    transmitted = power(substrate, result.transmission(polarization)) / normalization
    return reflected, transmitted

## 5. Check the familiar matching film

At normal incidence, a quarter-wave film cancels the reflection at its design wavelength when its index is the geometric mean of the surrounding indices. Bare glass, an ideal matching layer, and a magnesium-fluoride layer are three cases of the same constitutive construction. Zero film thickness gives the bare interface. The fixed indices here describe lossless, nondispersive materials.

In [ ]:
# Shared wavelengths and illumination; all lengths are in nm.
design_wavelength = 550.0
wavelength_samples = 401
wavelengths = np.linspace(450, 850, wavelength_samples)
incident_index = 1.0

In [ ]:
coating_names = ("Bare glass", "Ideal matching layer", "MgF₂ layer")
coating_indices = np.array([substrate_index, np.sqrt(incident_index * substrate_index), 1.38])
optical_thicknesses = np.array([0.0, design_wavelength / 4, design_wavelength / 4])
coating_thicknesses = optical_thicknesses / coating_indices

incident = isotropic_ports(incident_index, parallel)
substrate = ports_from_medium(substrate_medium, substrate_index, parallel)
chi = dielectric(coating_indices[:, None] ** 2 * np.ones(3), reluctivity)
medium = medium_from_chi(chi, parallel)
distance = 2 * np.pi * coating_thicknesses[:, None] / wavelengths
coating = propagate(medium.generator[:, None], distance)           # [coatings, wavelengths] Boundary <- Boundary
result = scatter(coating(substrate.outgoing), incident)
matching_reflection, matching_transmission = powers(result, mv.x, incident, substrate)
save_figure(render.spectra(wavelengths, matching_reflection, coating_names), "coating_chi_matching");

## 6. Turn the material axes through the coating

A birefringent crystal responds differently along its optic axis and across it. Turn that axis slightly from layer to layer by transporting the entire constitutive map with a rotor. Both helices below use the same material and thickness; only the sense of the rotor changes. The rods show one helix's unoriented optic axes. The stack samples six full turns, with sixteen homogeneous slices per turn.

In [ ]:
ordinary_index = 1.5
extraordinary_index = 1.8
surrounding_index = (ordinary_index + extraordinary_index) / 2
pitch = 400.0                                                     # nm per full director turn.
turns = 6
slices_per_turn = 16
layers_count = turns * slices_per_turn
thickness = pitch / slices_per_turn
twist_signs = np.array([1.0, -1.0])
permittivity = np.array([extraordinary_index ** 2, ordinary_index ** 2, ordinary_index ** 2])

depths = (np.arange(layers_count) + 0.5) * thickness
angles = 2 * np.pi * depths[:, None] / pitch * twist_signs
rotors = (mv.xy * (angles / 2)).exp()                              # [layers, helices] Rotor
directors = rotors >> mv.x                                         # [layers, helices] Vector
centres = mv.z * depths                                            # [layers] Vector
crystal = dielectric(permittivity, reluctivity)
# Carry an input field into the crystal frame, apply chi, and carry its output back.
chi = rotors >> crystal(rotors << Bivector)                        # [layers, helices] Antibivector <- Bivector
medium = medium_from_chi(chi, parallel)
save_figure(render.twist(centres, directors[:, 0]), "coating_twist");

## 7. Select the incident handedness

The rotating response couples the two transverse polarizations and produces a reflection band for one circular input. Reverse the material's twist and the two reflection spectra exchange: the left panel is the positive twist, the right the negative. The inputs are labelled by their actual complex electric vectors, fixing the handedness convention together with the positive time phase. This selective reflection is used in [cholesteric optical reflectors](https://www.nature.com/articles/nmat2045).

In [ ]:
polarization_signs = np.array([1.0, -1.0])
polarization_names = ("x + i y", "x − i y")

distance = 2 * np.pi * thickness / wavelengths
layers = propagate(medium.generator[..., None], distance)         # [layers, helices, wavelengths] Boundary <- Boundary
propagation = compose(layers)                                      # [helices, wavelengths] Boundary <- Boundary
surrounding = isotropic_ports(surrounding_index, parallel)
result = scatter(propagation(surrounding.outgoing), surrounding)
polarizations = (mv.x + mv.y * (1j * polarization_signs)) / np.sqrt(2)
twisted_reflection, twisted_transmission = powers(result, polarizations[:, None, None],
                                                  surrounding, surrounding)   # [polarizations, helices, wavelengths] Scalar
save_figure(render.handedness(wavelengths, twisted_reflection, polarization_names), "coating_handedness");

## 8. Tilt the beam

The two circular inputs see different reflection bands. Tilting the beam shifts and reshapes those bands; both polarizations remain coupled in the same boundary map. The maps below show reflected power across wavelength and incidence angle for the positive material twist, for the $x + iy$ input on the left and the $x - iy$ input on the right.

In [ ]:
angle_samples = 41
angles = np.linspace(0, np.deg2rad(70), angle_samples)
map_wavelength_samples = 121
map_wavelengths = np.linspace(450, 850, map_wavelength_samples)

parallel_oblique = mv.t + mv.x * (surrounding_index * np.sin(angles))
# Keep one material twist and batch over the illumination angles.
oblique_medium = medium_from_chi(chi[:, 0, None], parallel_oblique)
distance = 2 * np.pi * thickness / map_wavelengths
# One layer at a time, each batched over angles and wavelengths.
oblique_propagation = compose(propagate(generator[:, None], distance) for generator in oblique_medium.generator)
oblique_ports = isotropic_ports(surrounding_index, parallel_oblique[:, None])
oblique_result = scatter(oblique_propagation(oblique_ports.outgoing), oblique_ports)
# Rotate the actual electric circles with the ray, then take their tangential traces.
tilt = (mv.zx * (angles / 2)).exp()
oblique_field = tilt[None, :] >> polarizations[:, None]
tangential_field = -mv.z | (mv.z ^ oblique_field)
angle_reflection, angle_transmission = powers(oblique_result, tangential_field[..., None],
                                             oblique_ports, oblique_ports)
save_figure(render.angular(map_wavelengths, angles, angle_reflection), "coating_polarization_map");

## 9. Follow the field inside

At the centre of the reflection band, one circular input forms a standing wave that fades into the coating; the other continues travelling through it. These are the actual electric fields reconstructed from χ, shown over two optical cycles, for the $x + iy$ input on the left and the $x - iy$ input on the right. The fading is reflection, not absorption.

In [ ]:
field_wavelength = pitch * surrounding_index
samples_per_layer = 5
fractions = np.linspace(0, 1, samples_per_layer)
field_depths = (np.arange(layers_count)[:, None] + fractions) * thickness
field_medium = medium_from_chi(chi[:, 0], parallel)
field_layers = propagate(field_medium.generator,
                         np.full(layers_count, 2 * np.pi * thickness / field_wavelength))
field_result = scatter(compose(field_layers)(surrounding.outgoing), surrounding)
exit_state = surrounding.outgoing(field_result.transmission(polarizations))   # [cases] Boundary

# Each layer's right face holds the exit state carried back through every layer after it.
faces = accumulate(field_layers[:0:-1], lambda state, crossing: crossing(state), initial=exit_state)
right_faces = stack(tuple(faces)[::-1], axis=-1)                   # [cases, layers] Boundary
# From each right face back to the samples inside its layer, all layers at once.
remaining = 2 * np.pi * thickness * (1 - fractions) / field_wavelength
inward = propagate(field_medium.generator[:, None], remaining)     # [layers, samples] Boundary <- Boundary
# Reconstruct physical bivectors from the boundary states, starting from the transmitted field.
interior = field_medium.reconstruct[:, None](inward(right_faces[..., None]))   # [cases, layers, samples] Bivector
electric_field = mv.t | interior
save_figure(render.field(field_depths, electric_field), "coating_internal_field");

## 10. Step real fields through the helix

So far each wavelength was followed on its own, its phase held in complex coefficients. Stepping the fields themselves through time needs no phase. The same constitutive maps act on real field bivectors at the nodes of a line through the helix, and both exterior Maxwell equations, $d\wedge F = 0$ for the field $F$ and $d\wedge G = 0$ for its excitation $G$, become a leapfrog. Electric planes sit on the nodes and magnetic planes halfway between them. Contracting with the observer `t` undoes the time part of each equation's wedge, so each update is a contraction of a difference along `z`. The difference between neighbouring nodes is a sparse map over the line; its reverse runs each difference back, and the two are the adjoint pair under which the leapfrog keeps its energy.

In [ ]:
# Real coefficients: the time steps carry the phase.
real = NumpyContext(ga).multivector
spacing = 40.0                                                     # nm between nodes
cells = 1000
slab_start = 18000.0                                               # nm
positions = np.arange(cells) * spacing
inside = ((positions >= slab_start) & (positions < slab_start + turns * pitch)).astype(float)
surrounding_chi = dielectric(np.full(3, surrounding_index ** 2), reluctivity)
node_rotors = (mv.xy * (np.pi * (positions - slab_start) / pitch)).exp()
node_chi = (inside * (node_rotors >> crystal(node_rotors << Bivector))
            + (1 - inside) * surrounding_chi).real()                # [nodes] Antibivector <- Bivector
node_inverse = node_chi.inverse()                                  # [nodes] Bivector <- Antibivector
# Only magnetic planes sit between the nodes, and their response is the same everywhere.
between_chi = surrounding_chi.real()
# The difference reads, at each node, the next node's value less its own.
sites = np.arange(cells)
difference = SparseExtensor(real.scalar(np.tile([[1.0], [-1.0]], (cells, 1))), np.repeat(sites, 2),
                            np.stack([(sites + 1) % cells, sites], axis=-1).reshape(-1), (cells, cells))   # [nodes, nodes] Scalar cells


def evolve(electric: Bivector, magnetic: Bivector, interval: float, count: int):
    rate = interval / spacing
    for _ in range(count):
        # d ^ field == 0: the magnetic planes move with the electric planes on either side.
        magnetic = magnetic - (real.t | (real.z ^ (difference * electric.field()).batch())) * rate
        # d ^ excitation == 0: the excitation's spatial planes move with that of the magnetic planes
        # on either side, and the inverse response turns them back into electric planes.
        excitation = between_chi(magnetic)
        displacement = (real.t | (real.z ^ (~difference * excitation.field()).batch())) * rate
        electric = electric + node_inverse(displacement)
        yield electric, magnetic


def energy(electric: Bivector, magnetic: Bivector) -> Scalar:
    # Magnetic planes pair with their excitation to minus their energy.
    return ((electric & node_chi(electric)) - (magnetic & between_chi(magnetic))) * (spacing / 2)


A pulse at the band's centre, turning with the helix, against it, or not at all: the third is the average of the first two. Its magnetic planes follow from its electric planes for a wave already travelling towards `+z`. The animation follows the pulse that does not turn. Its field traces a flat zigzag on the way in; the part turning with the helix comes back, and the part turning against it passes through, each tracing loops of its own sense. The colour is the field's direction around the axis.

In [ ]:
pulse_start = 10000.0                                              # nm, the pulse's centre
pulse_width = 2000.0                                               # nm, its envelope's standard deviation
hands = np.array([1.0, -1.0])                                      # turning with and against the helix
courant = 1.0                                                      # time step per node spacing
duration = 27000.0                                                 # nm of light travel
frame_count = 120
interval = courant * spacing


def pulse(at: np.ndarray) -> Bivector:
    envelope = np.exp(-((at - pulse_start) / pulse_width) ** 2 / 2)
    phases = 2 * np.pi * surrounding_index * at / field_wavelength
    turned = (real.xy * (hands[:, None] * phases / 2)).exp() >> real.x       # [hands, nodes] Vector
    return (envelope * stack([turned[0], turned[1], turned.sum(axis=0) / 2])) ^ real.t


electric_planes = pulse(positions)                                 # [cases, nodes] Bivector
# Travelling towards +z: the magnetic planes half a node ahead and half a step earlier.
magnetic_planes = surrounding_index * (real.t | (real.z ^ pulse(positions + spacing / 2 + interval / (2 * surrounding_index))))
steps = round(duration / interval)
stride = steps // frame_count
states = list(islice(evolve(electric_planes, magnetic_planes, interval, steps), stride - 1, None, stride))
pulses = stack([state for state, _ in states])                     # [frames, cases, nodes] Bivector

slab = np.flatnonzero(inside)
show_animation(render.pulse(real.z * positions, real.z * positions[slab],
                            (node_rotors >> mv.x)[slab].real(), pulses[:, 2]), "coating_pulse", 50)


The leapfrog keeps the pulse's energy. Once the pulses have left the helix, the energy in front of it is what came back, and the energy behind it what passed through. The pulse turning with the helix comes back almost entirely, as the reflectance band of section 7 says.

In [ ]:
before = energy(electric_planes, magnetic_planes).sum(axis=-1)     # [cases] Scalar
after = energy(*states[-1])                                        # [cases, nodes] Scalar
reflected = (after * (positions < slab_start)).sum(axis=-1) / before
transmitted = (after * (positions >= slab_start + turns * pitch)).sum(axis=-1) / before


In [ ]:
# Per case: turning with the helix, against it, and not at all.
with np.printoptions(precision=3, suppress=True):
    print("reflected:  ", reflected.to_array())
    print("transmitted:", transmitted.to_array())


## 11. Change the boundary response without changing the bulk speeds

An axion contribution adds a scalar multiple of the field itself to its excitation. When constant inside a material, it leaves the bulk propagation speeds unchanged. Its jump changes how the fields on either side satisfy the same continuity conditions, producing reflection into both the incident and perpendicular polarizations. Here the ordinary dielectric response is identical on both sides, so the [axion discontinuity](https://arxiv.org/abs/physics/0504172) supplies all the reflection.

In [ ]:
axion_samples = 241
axion_jumps = np.linspace(-3, 3, axion_samples)
axion_names = ("Parallel to incident", "Perpendicular to incident", "Total reflection")

chi = dielectric(np.full(3, surrounding_index ** 2), reluctivity)
incident = isotropic_ports(surrounding_index, parallel)
# The scalar extensor carries the jump batch into the open field map.
axion_chi = chi + mv.scalar(axion_jumps[:, None]) * Bivector
substrate_medium = medium_from_chi(axion_chi, parallel)
substrate = ports_from_medium(substrate_medium, surrounding_index, parallel)
# There is no film: the substrate's outgoing wave meets the entrance face as it leaves.
result = scatter(substrate.outgoing, incident)
reflected = result.reflection(mv.x)                               # [jumps] Polarization
parallel_reflection = -mv.x * (mv.x | reflected)
perpendicular_reflection = -mv.y * (mv.y | reflected)
axion_reflection = stack([
    power(incident, parallel_reflection),
    power(incident, perpendicular_reflection),
    power(incident, reflected),
]) / power(incident, mv.x)
save_figure(render.axion(axion_jumps, axion_reflection, axion_names), "coating_axion");

The layer reconstruction assumes an invertible longitudinal response and a diagonalizable propagation generator. The exterior media are lossless, isotropic and nonmagnetic, with propagating waves and a real axion term. The examples use real, wavelength-independent material parameters; the twisted coating is a sequence of homogeneous slices approximating a continuous helix.

In Berreman's four-component transfer-matrix notation, the boundary propagation is represented by a four-by-four map; its entries depend on the chosen tangential field coordinates. [Berreman's treatment](https://doi.org/10.1364/JOSA.62.000502) describes stratified anisotropic media and cholesteric liquid crystals in that notation.